In [0]:
!pip install pdfplumber
dbutils.library.restartPython()   # restart Python to pick up the new versions

In [0]:
import requests
import os
import re
import pdfplumber
import pandas as pd

In [0]:
all_players = []

with pdfplumber.open("/Volumes/workspace/default/football_data/2026_FIFA_World_Cup_squads.pdf") as pdf:
    for page in pdf.pages:
        table = page.extract_table()

        if table:
            df = pd.DataFrame(table)

            # Player name is usually column 2
            if len(df.columns) >= 3:
                names = df.iloc[1:, 2].dropna().tolist()
                all_players.extend(names)

In [0]:
def pdf_extract(str_pdf):
    page_title = str_pdf
    volume_path = "/Volumes/workspace/default/football_data"
    save_path = f"{volume_path}/{str_pdf}.pdf"
    headers = {
        "User-Agent": "DatabricksFootballBot/1.0 (your-email@example.com)"
    }

    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.football_data")

    url = f"https://en.wikipedia.org/api/rest_v1/page/pdf/{requests.utils.quote(page_title)}"

    response = requests.get(url, headers=headers, stream=True)

    if response.status_code == 200:
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"PDF saved to {save_path}")
    else:
        print(f"Error {response.status_code}: {response.text}")

In [0]:
for i in range(0,len(all_players)-1,1):
    pdf_extract(all_players[i])
    print(i)

In [0]:
len(all_players)